## 📥 LOAD

Carregue os dados brutos para o notebook.  
Geralmente, isso envolve ler um arquivo `.jsonl` ou `.csv` que contém os exemplos originais.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
dataset_amazon_titles = "/content/drive/MyDrive/DatasetAmazonTitles/trn.json"
amazon_titles_alpaca_format_path = "/content/drive/MyDrive/DatasetAmazonTitles/amazon_titles_alpaca_cleaned_v3.jsonl"

## 🧹 CLEANING

Realize a limpeza e o pré-processamento dos dados.  
Aqui, você pode aplicar funções para normalizar textos, remover exemplos duplicados e filtrar entradas indesejadas.

#🛠️ Bibliotecas e funcionalidades

- **re**: Expressões regulares para filtrar e ajustar textos.
- **ftfy**: Corrige caracteres bugados e encoding estranho.
- **cleantext**: Limpa o texto (remove links, e-mails, telefones, quebra de linha, etc).
- **langdetect**: Garante que só exemplos em inglês passem.

---

#⚙️ O que o código faz?

1. Limpa e padroniza o texto.
2. Filtra exemplos ruins (muito curtos, só números, frases promocionais, aspas, etc).
3. Só aceita exemplos em inglês.

---

🦙 No final, o retorno segue o estilo Alpaca:
```json
{"Instruction": "...", "Input": "...", "Output": "..."}
```

In [ ]:
import json
import random

instructions = [
    "Describe a product by title",
    "Describe this product",
    "Give a description of the product type",
    "Explain what this product is",
    "Provide details about this product",
    "Sumarize the type of this product"
    ]

seen_titles = set()

with open(dataset_amazon_titles, "r", encoding="utf-8") as f, open(amazon_titles_alpaca_format_path, "w", encoding="utf-8") as out:
    for line in f:
        data = json.loads(line)
        title = data.get("title")
        content = data.get("content")

        # Ignorar linhas inválidas
        if not title or not content:
            continue

        # Normalizar o título (por exemplo, removendo espaços extras e transformando em minúsculas)
        normalized_title = title.strip().lower()

        # Ignorar duplicados
        if normalized_title in seen_titles:
            continue

        seen_titles.add(normalized_title)

        dict_alpaca_style = {
            "Instruction": random.choice(instructions),
            "Input": title.strip(),
            "Output": content.strip()
        }

        out.write(json.dumps(dict_alpaca_style, ensure_ascii=False) + "\n")

In [ ]:
!head -n 10 /content/drive/MyDrive/DatasetAmazonTitles/amazon_titles_alpaca_cleaned_v3.jsonl

{"Instruction": "Describe this product", "Input": "Girls Ballet Tutu Neon Pink", "Output": "High quality 3 layer ballet tutu. 12 inches in length"}
{"Instruction": "Describe a product by title", "Input": "Mog's Kittens", "Output": "Judith Kerr&#8217;s best&#8211;selling adventures of that endearing (and exasperating) cat Mog have entertained children for more than 30 years. Now, even infants and toddlers can enjoy meeting this loveable feline. These sturdy little board books&#8212;with their bright, simple pictures, easy text, and hand&#8211;friendly formats&#8212;are just the thing to delight the very young. Ages 6 months&#8211;2 years."}
{"Instruction": "Describe this product", "Input": "Girls Ballet Tutu Neon Blue", "Output": "Dance tutu for girls ages 2-8 years. Perfect for dance practice, recitals and performances, costumes or just for fun!"}
{"Instruction": "Provide details about this product", "Input": "The Prophet", "Output": "In a distant, timeless place, a mysterious prophet 

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=amazon_titles_alpaca_format_path, split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
!pip install ftfy clean-text langdetect html --quiet

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
import re
import ftfy
from cleantext import clean as clean_text
from langdetect import detect

# 4. Funções de pré-processamento
def preprocess(text: str) -> str:
    if not text:
        return ""
    text = ftfy.fix_text(text)            # Corrigir encoding bugado
    text = clean_text(                    # Limpeza geral
        text,
        fix_unicode=True,
        no_urls=True,
        no_emails=True,
        no_phone_numbers=True,
        no_line_breaks=True,
        lower=False,
    )
    text = re.sub(r"\s+", " ", text)      # Normalizar espaços
    return text.strip()

def clean(example):
    title = preprocess(example["Input"])
    desc  = preprocess(example["Output"])

    # Regras de Input (título)
    if len(title) < 3 or re.match(r"^[0-9\s\-_]+$", title):
        return None

    # Regras de Output (descrição)
    if len(desc) < 10 or len(desc) > 1000:
        return None
    if desc.startswith('"') and desc.endswith('"'):
        return None
    if re.search(r"(BUY NOW|FREE SHIPPING)", desc.upper()):
        return None

    # Linguagem (descartar não-inglês)
    try:
        if detect(desc) != "en":
            return None
    except:
        return None

    return {"Instruction": example["Instruction"], "Input": title, "Output": desc}

In [ ]:
dataset_clean = dataset.map(clean).filter(lambda x: x is not None)

Map:   0%|          | 0/1349470 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1349470 [00:00<?, ? examples/s]

In [ ]:
%pip install html

  Using cached html-1.16.tar.gz (7.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
import html

def fix_entities(example):
    example["Input"] = html.unescape(html.unescape(example["Input"]))
    example["Output"] = html.unescape(html.unescape(example["Output"]))
    return example

dataset_clean = dataset_clean.map(fix_entities)

Map:   0%|          | 0/1349470 [00:00<?, ? examples/s]

# 🚀 PUBLISH

Exporte os dados limpos no formato desejado.  
No contexto Alpaca, cada exemplo terá os campos "Instruction", "Input" e "Output", prontos para treinar modelos de linguagem.

In [ ]:
from datasets import DatasetDict

dataset_split = dataset_clean.train_test_split(test_size=0.1, seed=42)

dataset_split = DatasetDict({
    "train": dataset_split["train"],
    "validation": dataset_split["test"]
})

In [ ]:
dataset_split = dataset_clean.train_test_split(test_size=0.1, seed=42)

In [ ]:
from huggingface_hub import notebook_login
from google.colab import userdata

huggingface_token = userdata.get('huggingface_token')
dataset_split.push_to_hub("guillherms/amazon_titles_alpaca_cleaned_v3", token= huggingface_token)

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/608 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   0%|          |  525kB /  285MB            

...

Creating parquet from Arrow format:   0%|          | 0/135 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   6%|5         | 3.68MB / 63.2MB            

CommitInfo(commit_url='https://huggingface.co/datasets/guillherms/amazon_titles_alpaca_cleaned_v3/commit/0b0bd91e6ab354424012f50fb4761282848bb298', commit_message='Upload dataset', commit_description='', oid='0b0bd91e6ab354424012f50fb4761282848bb298', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/guillherms/amazon_titles_alpaca_cleaned_v3', endpoint='https://huggingface.co', repo_type='dataset', repo_id='guillherms/amazon_titles_alpaca_cleaned_v3'), pr_revision=None, pr_num=None)